In [0]:


####https://cursos.alura.com.br/course/testes-estatisticos-comparacao/task/20934

Este projeto aplica métodos de estatística avançada para comparar o engajamento de usuários em diferentes trailers de filmes em uma plataforma de streaming. 

A análise utiliza amostras de usuários e o tempo médio de visualização como métrica de engajamento. Inicialmente, é realizada a comparação entre dois grupos usando o teste t para médias. Em seguida, com a inclusão de um terceiro trailer, aplica-se ANOVA para comparar múltiplos grupos.

O objetivo é demonstrar como técnicas de inferência estatística podem ser usadas para apoiar decisões baseadas em dados.



1. **ANOVA One-Way:** Descobrir se há alguma diferença estatisticamente significativa entre as médias de visualização dos três trailers (Drama, Comédia e Futurista).
2. **Teste de Tukey HSD:** Se a ANOVA nos der um sinal verde, usaremos este teste para descobrir **onde** as diferenças se encontram.
3. **Teste de Bonferroni:** Exploraremos uma alternativa ao Tukey, entendendo suas vantagens e desvantagens


In [0]:
pip install pingouin

In [0]:
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.stats.anova as anova
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import statsmodels.stats.anova as anova
import scipy.stats as stats
from statsmodels.stats.anova import AnovaRM
import pingouin as pg

In [0]:
df = pd.read_csv("/Workspace/Repos/patriciatamiresdesousa@gmail.com/estatistica-avancada-comparacao-entre-grupos/trailers_dados_aulas_1_e_2.csv")
df.head()

In [0]:

## Visualizacao dos dados dos trailers

trailer_types = df['trailer_tipo'].unique()
fig, axes = plt.subplots(1, len(trailer_types), figsize=(15, 5))
for i, trailer_type in enumerate(trailer_types):
    subset_df = df[df['trailer_tipo'] == trailer_type]
    sns.histplot(subset_df['tempo_visualizacao'], ax=axes[i], kde=True)
    axes[i].set_title(f'Distribuição de Tempo de Visualização - {trailer_type}')
    axes[i].set_xlabel('Tempo de Visualização')
    axes[i].set_ylabel('Frequência')
y_max = max([ax.get_ylim()[1] for ax in axes])
for ax in axes:
    ax.set_ylim(0, y_max)
    
x_min = df['tempo_visualizacao'].min()
x_max = df['tempo_visualizacao'].max()
for ax in axes:
    ax.set_xlim(x_min, x_max)

plt.tight_layout()
plt.show()

In [0]:
#### Tabela estatistica
### tanto a media como o desvio padrao esta altas para o trailer futurista 


tabela_estat = df.groupby('trailer_tipo')['tempo_visualizacao'].agg(['mean', 'std', 'count']).reset_index()
tabela_estat.rename(columns={'mean': 'media', 'std': 'desvio_padrao', 'count': 'numero_observacoes'}, inplace=True)
display(tabela_estat)

In [0]:
##Tabela ANOVA


##Resulto F grande o que siginifica que a diferença entre os trailer
## Aqui como avalio a diferença, o valor F avalia a diferenca eentre os grupos sendo essa diferença parecida ou não
## quando as médias são proximas o valor F é baixo, quando as médias são altas os valores são altos o que siginifca uma diferença entre os trailers
##Outro ponto importante o P-Value(>P) quando menor maior a diferenca siginfiicativa dos grupos, quando maior, maior a diferença pode ser aleatoria. Aqui a diferença é de 1.63e-19 = 1,63x10-19
###0.000000000000000000163 muito menor que 0.05


modelo = smf.ols('tempo_visualizacao ~ C(trailer_tipo)', data=df).fit()
anova_tabela = anova.anova_lm(modelo, typ=2)

print(anova_tabela)

In [0]:

###Visualizando a distribuição F

df_dados_anova = anova_tabela.loc['C(trailer_tipo)', 'df']
df_resido = anova_tabela.loc['Residual', 'df']


x = np.linspace(f.ppf(0.001, df_dados_anova, df_resido),
                f.ppf(0.999, df_dados_anova, df_resido), 100)
                

plt.figure(figsize=(10, 6))
plt.plot(x, f.pdf(x, df_dados_anova, df_resido),
         'r-', lw=2, alpha=0.6, label=f'df1={int(df_dados_anova)}, df2={int(df_resido)}')

f_calculated = anova_tabela.loc['C(trailer_tipo)','F']
plt.axvline(f_calculated, color='blue', linestyle='dashed', linewidth=2, label=f'F Calculado = {f_calculated:.2f}')


plt.title('Distribuição F com Graus de Liberdade do Teste ANOVA')
plt.xlabel('Valor de F')
plt.ylabel('Densidade de Probabilidade')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

####Teste de Tukey 

##### Um teste estatístico usado depois de uma ANOVA para descobrir quais grupos são diferentes entre si.

In [0]:
#### Teste Tukey= Objetivo é encontrar qual grupo tem diferença 
### pairwise_tukeyhsd aplica o teste enqaunto o parametro alpha é definifo para representar a significação do teste 
#### p-value 0.0964 > 0.05 = Não existe diferença estatisticamente significativa entre Comédia e Drama no tempo de visualização
### Já o filme Comedia vs Futurista  p-value ≈ 0 = Trailers futuristas têm tempo de visualização muito maior que comédia

teste_tukey = pairwise_tukeyhsd(endog=df['tempo_visualizacao'], groups=df['trailer_tipo'], alpha=0.05)
print(teste_tukey.summary())

####Teste de Bonferroni

#####Um método usado em Estatística para corrigir o p-value quando fazemos várias comparações ao mesmo tempo

#####Quando é aplicado muitos testes aumenta a chance de encontrr falsos posistivos. Por exemplo eu aplico 20 teste e os 20 tem resultado = 0.5 mesmo o teste não aparenta uma diferença real estatisticamente pode existir 1 teste significativo por acaso, chamam isso de erro tipo I. Ideia do teste de Bonferroni é tornar o teste mais rigoroso

In [0]:
### Teste T independente (Fazendo a comparação dudas médias)

groupos = df['trailer_tipo'].unique()
p_values = []
comparisons = []

for i in range(len(groupos)):
    for j in range(i + 1, len(groupos)):
        group1 = groupos[i]
        group2 = groupos[j]
        data1 = df[df['trailer_tipo'] == group1]['tempo_visualizacao']
        data2 = df[df['trailer_tipo'] == group2]['tempo_visualizacao']

In [0]:
### Teste T para cada par de grupo
t_stat, p_value = stats.ttest_ind(data1, data2)

In [0]:
## corriginodo o nive de siginificancia e apresentando os resultados dos testes
p_values.append(p_value)
comparisons.append(f'{group1} vs {group2}')

reject, pvals_corrected, _, _ = multipletests(p_values, method='bonferroni')

In [0]:
# Resultados dos testes 
### P valor está e notacao cientifica mas siginfiica <.5 o que da uma diferença significativa 
### Mesmo com a correção do Bonferroni continua bem siginfiicativo 
## Rejeicao true siginifica que a hipotese nula foi rejeitada

### Resumindo o tempo de visualização de trailers Futuristas é significativamente diferente do de Comédia.

bonferroni_results = pd.DataFrame({
    'Comparação': comparisons,
    'P-valor Original': p_values,
    'P-valor Corrigido (Bonferroni)': pvals_corrected,
    'Rejeitar H0': reject
})

display(bonferroni_results)

###### ANOVA - Para medidas repetidas

In [0]:
df_dados_sobre_ansiedade = pd.read_csv("/Workspace/Repos/patriciatamiresdesousa@gmail.com/estatistica-avancada-comparacao-entre-grupos/ansiedade_data.csv")
df_dados_sobre_ansiedade.head()

In [0]:
df_dados_sobre_ansiedade.shape

In [0]:
# Transformando o DataFrame 'df_dados_sobre_ansiedade' para o formato 'long'
# Usando as colunas 'Paciente', 'inicio', '3meses', '6meses'
data_long = pd.melt(df_dados_sobre_ansiedade,
                    id_vars=['Paciente'],
                    value_vars=['inicio', '3meses', '6meses'],
                    var_name='Tempo',
                    value_name='Escore_Ansiedade')

# Convertendo a coluna 'Tempo' para o tipo categórico ordenado
data_long['Tempo'] = pd.Categorical(data_long['Tempo'],
                                    categories=['inicio', '3meses', '6meses'],
                                    ordered=True)

In [0]:
### Analise descritiva dos dados dos pacientes 

desc_estataistica_ansiedade = data_long.groupby('Tempo')['Escore_Ansiedade'].agg(['count', 'mean', 'median','std', 'min', 'max'])
desc_estataistica_ansiedade

In [0]:
# Gráfico de Linha com Média e Erro Padrão

sns.pointplot(
    data=data_long,
    x='Tempo',
    y='Escore_Ansiedade',
    errorbar='se',
    capsize=0.1,
    color='cornflowerblue'
)

plt.title('Nível de Ansiedade ao Longo do Tempo (Média e Erro Padrão)')
plt.ylabel('Nível de Ansiedade')
plt.xlabel('Tempo')
plt.grid(True)
plt.show()

print("\n")

In [0]:
# Usando o DataFrame df_long_ansiedade e os nomes de colunas corretos
aov_stats = AnovaRM(
    data= data_long,
    depvar='Escore_Ansiedade', # Variável dependente
    subject='Paciente',        # Coluna que identifica os sujeitos
    within=['Tempo'],          # Variável "within-subject" (medidas repetidas)
    aggregate_func='mean'    # Função de agregação, se necessário (geralmente 'mean' para RM ANOVA)
)

model_results = aov_stats.fit()
print(model_results)
print("\n")



###Resultados apresentados nessa tabela, resulta no valor da estitisca F, número de graus de liberdade e probabilidade do p-valor. 
### O p-valor é muito pequeno, menos que o nivél de siginificandia de 0,5 (0,5 é um padrão estatistica dentro das pesquisas, seja com ML, seja em pyhton)
#### Indicando que as médias dos escores de ansiedade dos pacientes ao longo do tempo são estatisticamente diferentes em pelo menos dois momentos.

In [0]:
### Quando usado a ANOVA para medidas repetidas, é ecessario fazer o teste de esfericidade para garantir que não tenha conclusões erradas
### No teste de repetições eu estou comparando os mesmo individuos, em condições diferente e isso pode gerar uma dependencia entre os dados
### Então o teste de esfericidade é usado para verificar se os dados são independentes ou não
### Se o teste de esfericidade for significativo, isso indica que os dados não são independentes e pode afetar o resultado da AN
## Com isso se aplica o Pingoun ele ajusta automaticamente os dados para que sejam independentes

aov_pingouin = pg.rm_anova(
    data=data_long,
    dv='Escore_Ansiedade', # Variável dependente
    within='Tempo', # Variável "within-subject" (medidas repetidas)
    subject='Paciente', # Coluna que identifica os sujeitos
    detailed=True, # Incluir detalhes, como o teste de esfericidade
    effsize='np2' # Incluir tamanho do efeito (eta-quadrado parcial)
)

display(aov_pingouin)
print("\n")